# Testing how to calculate the Gauss-Newton vector product in Jax 

I believe that with repeated calls, it's better to use jax.linearize and jax.linear_transpose but I need to learn how to call these. 

In [80]:
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update('jax_default_matmul_precision', 'highest')
import jax.numpy as jnp
import flax.linen as nn
from flax.training import train_state
import optax
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
# For viz
import treescope
treescope.basic_interactive_setup(
    autovisualize_arrays=True,
    abbreviation_threshold=2, 
)
treescope.register_as_default()


In [129]:
# Create standard model
class MLP_1D(nn.Module):
    @nn.compact
    def __call__(self, x):
        x = nn.Dense(
            features=100, use_bias=True,
            kernel_init=nn.initializers.kaiming_uniform(),  # Kaiming uniform initialization
            bias_init=nn.initializers.ones  # Bias initialized to zero
        )(x)
        x = nn.Dense(
            features=100, use_bias=True,
            kernel_init=nn.initializers.kaiming_uniform(),  # Kaiming uniform initialization
            bias_init=nn.initializers.ones  # Bias initialized to zero
        )(jnp.tanh(x))
        x = nn.Dense(
            features=2, use_bias=True,
            kernel_init=nn.initializers.kaiming_uniform(),  # Kaiming uniform initialization
            bias_init=nn.initializers.ones  # Bias initialized to zero
        )(jnp.tanh(x))

        return x

In [130]:
key = jax.random.PRNGKey(0)
model = MLP_1D()
params = model.init(key, jnp.ones((1, 1)))

In [131]:
params

{'params': {'Dense_0': {'kernel': <jax.Array float32(1, 100) ≈0.052 ±1.4 [≥-2.4, ≤2.3] nonzero:100
     <Arrayviz rendering>
   | Device: GPU 0>,
   'bias': <jax.Array float32(100,) ≈1.0 ±0.0 [≥1.0, ≤1.0] nonzero:100
     <Arrayviz rendering>
   | Device: GPU 0>},
  'Dense_1': {'kernel': <jax.Array float32(100, 100) ≈-0.0003 ±0.14 [≥-0.24, ≤0.24] nonzero:10_000
     <Arrayviz rendering>
   | Device: GPU 0>,
   'bias': <jax.Array float32(100,) ≈1.0 ±0.0 [≥1.0, ≤1.0] nonzero:100
     <Arrayviz rendering>
   | Device: GPU 0>},
  'Dense_2': {'kernel': <jax.Array float32(100, 2) ≈0.0016 ±0.14 [≥-0.24, ≤0.24] nonzero:200
     <Arrayviz rendering>
   | Device: GPU 0>,
   'bias': <jax.Array float32(2,) ≈1.0 ±0.0 [≥1.0, ≤1.0] nonzero:2
     <Arrayviz rendering>
   | Device: GPU 0>}}}

In [133]:
x = jnp.arange(100).reshape((100, 1))
model.apply(params, x)

<jax.Array float32(100, 2) ≈0.57 ±0.42 [≥0.086, ≤1.9] nonzero:200
  <Arrayviz rendering>
| Device: GPU 0>

In [134]:
# Create a fake gradient 
def generate_random_pytrees(original_pytree, num_trees, key):
    def randomize_leaf(leaf, key):
        # Generate random values with the same shape as the leaf
        return jax.random.uniform(key, shape=leaf.shape, dtype=leaf.dtype)

    random_pytrees = []
    for i in range(num_trees):
        # Split the key for reproducibility
        key, subkey = jax.random.split(key)
        # Generate a random pytree by mapping the randomize_leaf function
        random_pytree = jax.tree.map(lambda leaf: randomize_leaf(leaf, subkey), original_pytree)
        random_pytrees.append(random_pytree)

    return random_pytrees

fake_gradients = generate_random_pytrees(params, 10, jax.random.PRNGKey(1))

In [137]:
%%timeit
# Test with jax.jvp/jax.vjp; want to do J^TJ 
# @jax.jit
def compute_vjp(params, x):
    _, f_vjp = jax.vjp(lambda params: model.apply(params, x), params)
    return f_vjp
f_vjp = compute_vjp(params, x)

for grad in fake_gradients:
    out_tangent = f_vjp(
        jax.jvp(lambda params: model.apply(params, x), (params,), (grad,))[1]
    )
out_tangent

182 ms ± 196 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [139]:
# %%timeit
def compute_jvp_and_vjp(params, x):
    _, f_jvp = jax.linearize(lambda p: model.apply(p, x), params)
    f_vjp = jax.linear_transpose(f_jvp, params)
    return f_jvp, f_vjp

# Compute JVP and VJP
f_jvp, f_vjp = compute_jvp_and_vjp(params, x)

for grad in fake_gradients:
    out_tangent = f_vjp(f_jvp(grad))

In [141]:
len(out_tangent)

1